# Interactive & Distributed TPU Training with JAX on GKE Workspaces

This notebook demonstrates how easy it is to develop interactively on **1 TPU or 4 TPUs** using Cloud TPU v5e, and seamlessly scale out to a **distributed multi-host TPU slice** using the **Kubeflow Trainer Python SDK**—reusing the exact same JAX training function!

## 0. Install JAX for TPU

The default `jupyterlab` WorkspaceKind registered by `deploy_standalone.sh` runs the TPU image option on the upstream **CPU** base image, which has no `jax[tpu]`/`libtpu` (the Kubeflow SDK is already included). The next cell installs JAX for TPU with `--user` (into `~/.local` on the home volume, so they persist across restarts). On the custom TPU image from `images/build.sh` it is a quick no-op.

The cell also adds `~/.local` to `sys.path` so the packages are importable without a kernel restart. If JAX was already imported in this kernel before installing, restart the kernel (**Kernel → Restart Kernel...**) and continue from the device check.

In [ ]:
# Step 0 — same jax version as images/jupyterlab/requirements-tpu.txt.
%pip install --user --quiet "jax[tpu]==0.10.2"
# ~/.local/.../site-packages is only on sys.path if it existed at kernel start; add it now.
import site; site.addsitedir(site.getusersitepackages())

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp

# Verify connected TPU devices in the current interactive workspace
devices = jax.devices()
local_devices = jax.local_devices()
print(f"JAX Backend: {jax.default_backend()}")
print(f"Available devices ({len(devices)} total, {len(local_devices)} local): {devices}")

## 1. Synthetic Dataset Generation

We generate a synthetic 10-class image classification dataset (784 features per sample, matching Fashion-MNIST / MNIST dimensions).

In [ ]:
def generate_synthetic_data(num_samples=8192, num_features=784, num_classes=10, seed=42):
    np.random.seed(seed)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    labels_one_hot = np.eye(num_classes, dtype=np.float32)[labels]
    return images, labels_one_hot, labels

train_x, train_y_one_hot, train_y = generate_synthetic_data(num_samples=8192, seed=1)
test_x, test_y_one_hot, test_y = generate_synthetic_data(num_samples=1024, seed=2)

print(f"Train shapes: images {train_x.shape}, one-hot labels {train_y_one_hot.shape}")
print(f"Test shapes:  images {test_x.shape}, one-hot labels {test_y_one_hot.shape}")

## 2. Model Architecture & Forward Pass

We define a 3-layer MLP (`784 -> 256 -> 128 -> 10`) with ReLU activations and cross-entropy loss.

In [ ]:
def init_network_params(layers, key):
    keys = jax.random.split(key, len(layers) - 1)
    params = []
    for k, (in_dim, out_dim) in zip(keys, zip(layers[:-1], layers[1:])):
        w_key, b_key = jax.random.split(k)
        scale = jnp.sqrt(2.0 / in_dim)
        w = jax.random.normal(w_key, (in_dim, out_dim)) * scale
        b = jnp.zeros((out_dim,))
        params.append({"w": w, "b": b})
    return params

def forward(params, x):
    activations = x
    for layer in params[:-1]:
        activations = jax.nn.relu(jnp.dot(activations, layer["w"]) + layer["b"])
    final_layer = params[-1]
    return jnp.dot(activations, final_layer["w"]) + final_layer["b"]

def loss_fn(params, x, y_one_hot):
    logits = forward(params, x)
    log_preds = jax.nn.log_softmax(logits)
    return -jnp.mean(jnp.sum(log_preds * y_one_hot, axis=-1))

def compute_accuracy(params, x, targets):
    logits = forward(params, x)
    preds = jnp.argmax(logits, axis=-1)
    return jnp.mean(preds == targets)

params = init_network_params([784, 256, 128, 10], jax.random.PRNGKey(42))
print(f"Initialized 3-layer MLP. Layers: {[p['w'].shape for p in params]}")
print(f"Initial Loss: {loss_fn(params, train_x[:128], train_y_one_hot[:128]):.4f}")

## 3. Reusable JAX Training Function (`train_mlp`)

The training function below is **100% reusable**:
- Runs interactively on local **1 TPU or 4 TPUs** inside your workspace.
- Runs distributed on a **multi-host TPU slice (8+ TPUs)** via the Kubeflow Trainer Python SDK, automatically discovering `jax-distributed` coordinator parameters from the environment.

In [ ]:
from functools import partial

def train_mlp(epochs=5, global_batch_size=256, lr=0.05):
    """Universal training function executed locally or distributed across multi-host TPUs."""
    import os
    import time
    import numpy as np
    import jax
    import jax.numpy as jnp
    from functools import partial

    # 1. Multi-host initialization (injected by jax-distributed runtime in Kubeflow TrainJobs)
    if "JAX_COORDINATOR_ADDRESS" in os.environ:
        import jax.distributed as dist
        raw_pid = os.environ.get("JAX_PROCESS_ID", "")
        if not raw_pid.isdigit():
            raw_pid = os.environ.get("JOB_COMPLETION_INDEX", "0")
        pid_val = int(raw_pid) if raw_pid.isdigit() else 0
        dist.initialize(
            coordinator_address=os.environ["JAX_COORDINATOR_ADDRESS"],
            num_processes=int(os.environ["JAX_NUM_PROCESSES"]),
            process_id=pid_val,
        )

    proc_idx = jax.process_index()
    num_devices = jax.local_device_count()
    global_devices = jax.device_count()
    print(f"[Host {proc_idx}] Initialized JAX: {num_devices} local devices, {global_devices} global devices", flush=True)

    # 2. Data & Model Setup
    num_samples = 8192
    num_features = 784
    num_classes = 10

    np.random.seed(42 + proc_idx)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    images = prototypes[labels] + np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    labels_one_hot = np.eye(num_classes, dtype=np.float32)[labels]

    def init_p(layers, key):
        keys = jax.random.split(key, len(layers) - 1)
        p_list = []
        for k, (in_d, out_d) in zip(keys, zip(layers[:-1], layers[1:])):
            w_k, b_k = jax.random.split(k)
            p_list.append({"w": jax.random.normal(w_k, (in_d, out_d)) * jnp.sqrt(2.0 / in_d), "b": jnp.zeros((out_d,))})
        return p_list

    def fwd(p, x):
        h = x
        for layer in p[:-1]:
            h = jax.nn.relu(jnp.dot(h, layer["w"]) + layer["b"])
        return jnp.dot(h, p[-1]["w"]) + p[-1]["b"]

    def l_fn(p, x, y):
        preds = jax.nn.log_softmax(fwd(p, x))
        return -jnp.mean(jnp.sum(preds * y, axis=-1))

    params = init_p([784, 256, 128, 10], jax.random.PRNGKey(42))
    replicated_params = jax.tree_util.tree_map(
        lambda x: jnp.broadcast_to(x, (num_devices,) + x.shape),
        params
    )

    # 3. Parallel step across all TPU cores using jax.pmap
    @partial(jax.pmap, axis_name="devices", in_axes=(0, 0, 0, None))
    def step(p, x, y, lr_rate):
        loss, grads = jax.value_and_grad(l_fn)(p, x, y)
        grads = jax.lax.pmean(grads, axis_name="devices")
        loss = jax.lax.pmean(loss, axis_name="devices")
        new_p = jax.tree_util.tree_map(lambda param, grad: param - lr_rate * grad, p, grads)
        return new_p, loss

    per_device_batch = max(1, global_batch_size // global_devices)
    batches = num_samples // (per_device_batch * num_devices)

    start_time = time.time()
    for ep in range(1, epochs + 1):
        t0 = time.time()
        perm = np.random.permutation(num_samples)
        x_shuffled = images[perm]
        y_shuffled = labels_one_hot[perm]
        total_loss = 0.0

        for b in range(batches):
            idx = b * (per_device_batch * num_devices)
            bx = x_shuffled[idx : idx + per_device_batch * num_devices].reshape(num_devices, per_device_batch, -1)
            by = y_shuffled[idx : idx + per_device_batch * num_devices].reshape(num_devices, per_device_batch, -1)
            replicated_params, l = step(replicated_params, bx, by, lr)
            total_loss += float(l[0])

        total_loss /= batches
        dur = time.time() - t0
        throughput = (batches * per_device_batch * global_devices) / dur
        if proc_idx == 0 or num_devices == global_devices:
            print(f"Epoch {ep:2d}/{epochs:2d} | Loss: {total_loss:.4f} | Throughput: {throughput:.0f} samples/sec ({dur:.2f}s)", flush=True)

    if proc_idx == 0:
        print(f"\n[Host {proc_idx}] Training complete in {time.time() - start_time:.2f}s across {global_devices} TPU cores!", flush=True)
    return replicated_params

print("Defined reusable train_mlp function.")

## 4. Run Interactive Local Training on Workspace TPUs

Execute `train_mlp` right here in the notebook. JAX automatically distributes training across all available TPU cores on this host (1 or 4 cores).

In [ ]:
# Train locally on the workspace's attached TPU cores
trained_params = train_mlp(epochs=5, global_batch_size=256, lr=0.05)

## 5. Model Evaluation & Sample Predictions

In [ ]:
# Evaluate using parameters from device 0
final_params = jax.tree_util.tree_map(lambda x: x[0], trained_params)
final_test_acc = compute_accuracy(final_params, test_x, test_y)
print(f"Final Test Accuracy: {final_test_acc * 100:.2f}%\n")

sample_inputs = test_x[:5]
sample_targets = test_y[:5]
sample_preds = jnp.argmax(forward(final_params, sample_inputs), axis=-1)

print("Predicted vs Actual Labels:")
for i in range(5):
    match = "OK" if sample_preds[i] == sample_targets[i] else "MISMATCH"
    print(f"  Sample {i+1}: Predicted = {int(sample_preds[i])}, True = {int(sample_targets[i])} [{match}]")

## 6. Scale to Multi-Host TPU Slice with Kubeflow Trainer Python SDK

Scale the **EXACT SAME** `train_mlp` function to a distributed multi-host TPU slice (2 hosts × 4 chips = 8 TPU v5e cores) using Kubeflow Trainer's Python SDK with the `jax-distributed` runtime—no YAML needed!

In [ ]:
import os
from kubeflow.trainer import CustomTrainer, TrainerClient
from kubeflow.trainer.options import kubernetes as k8s_options
from kubeflow.common.types import KubernetesBackendConfig

# 1. Initialize TrainerClient for tenant namespace
def _get_current_namespace():
    """Return DEMO_NAMESPACE if set, else the pod's mounted serviceaccount namespace, else 'default'."""
    if os.environ.get("DEMO_NAMESPACE"):
        return os.environ["DEMO_NAMESPACE"]
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        try:
            with open(ns_path) as f:
                ns = f.read().strip()
                if ns:
                    return ns
        except OSError:
            pass
    return "default"

ns = _get_current_namespace()
print(f"Using namespace: {ns}")
trainer_client = TrainerClient(backend_config=KubernetesBackendConfig(namespace=ns))

# 2. Pin worker pods to the multi-host TPU slice (2 hosts x 4 chips = 8 TPUs)
tpu_placement_patch = k8s_options.RuntimePatch(
    training_runtime_spec=k8s_options.TrainingRuntimeSpecPatch(
        template=k8s_options.JobSetTemplatePatch(
            spec=k8s_options.JobSetSpecPatch(
                replicated_jobs=[
                    k8s_options.ReplicatedJobPatch(
                        name="node",
                        template=k8s_options.JobTemplatePatch(
                            spec=k8s_options.JobSpecPatch(
                                template=k8s_options.PodTemplatePatch(
                                    spec=k8s_options.PodSpecPatch(
                                        node_selector={"cloud.google.com/compute-class": "tpu-v5-8-multi-host"},
                                        tolerations=[
                                            {"key": "google.com/tpu", "operator": "Exists", "effect": "NoSchedule"},
                                            {"key": "cloud.google.com/compute-class", "operator": "Exists", "effect": "NoSchedule"},
                                        ],
                                    ),
                                )
                            )
                        ),
                    )
                ]
            )
        )
    )
)

# 3. Submit distributed multi-host JAX TrainJob reusing train_mlp!
train_job = trainer_client.train(
    runtime="jax-distributed",
    trainer=CustomTrainer(
        func=train_mlp,
        image="us-docker.pkg.dev/cloud-tpu-images/jax-ai-image/tpu:latest",
        num_nodes=2,
        resources_per_node={"google.com/tpu": 4},
        env={
            "JAX_PLATFORMS": "tpu,cpu",
            "ENABLE_PJRT_COMPATIBILITY": "true",
        },
    ),
    options=[tpu_placement_patch],
)
print(f"Submitted multi-host TPU TrainJob: {train_job}")

In [ ]:
# 4. Stream training logs from both TPU hosts
print(f"Waiting for {train_job} to complete...")
trainer_client.wait_for_job_status(train_job, timeout=600)
print(f"\n=== Multi-Host Training Logs for {train_job} ===")
for log_line in trainer_client.get_job_logs(train_job):
    print(log_line)